# ViSongSearch: GPU experiments (dense, NA-FT, hybrid, reranking)

Notebook này chạy toàn bộ phần cần GPU của nghiên cứu ViSongSearch và xuất một file kết quả nhỏ để đưa lại vào project.

**Chuẩn bị (một lần):**
1. Tải `visongsearch-bundle.zip` (trong thư mục chung của project, mục `visongsearch/kaggle/`) lên Kaggle: *Datasets → New Dataset*, đặt tên `visongsearch-bundle`. Kaggle sẽ tự giải nén.
2. Mở notebook này trên Kaggle (*File → Import Notebook*), rồi *Add Input* → chọn dataset `visongsearch-bundle`.
3. *Settings*: **Accelerator = GPU T4 x2** (hoặc P100), **Internet = On** (tài khoản cần xác minh số điện thoại để bật Internet).
4. Bấm **Save Version → Save & Run All (Commit)** để chạy nền (tối đa 12 giờ), hoặc *Run All* nếu ngồi chờ.

**Khi xong:** vào tab *Output*, tải `visongsearch_results.zip` và gửi lại vào thread của project. Mọi lựa chọn mô hình được làm trên tập dev, không nhìn tập test.

Thời gian ước tính trên T4 với corpus pilot (548 bài): khoảng 40–70 phút.

In [ ]:
# 1. Find the bundle (Kaggle unzips uploaded datasets) and copy it to a writable folder
import glob, os, shutil, zipfile
ROOT = "/kaggle/working/vss"
if not os.path.exists(ROOT):
    inits = glob.glob("/kaggle/input/**/visong/__init__.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/visongsearch-bundle*.zip", recursive=True)
    if inits:
        shutil.copytree(os.path.dirname(os.path.dirname(inits[0])), ROOT)
    elif zips:
        zipfile.ZipFile(zips[0]).extractall(ROOT)
    else:
        raise SystemExit("Add the visongsearch-bundle dataset to this notebook first (Add Input).")
os.chdir(ROOT)
print(sorted(os.listdir(ROOT)))

In [ ]:
# 2. Libraries (torch and transformers come with the Kaggle image)
!pip install -q "sentence-transformers>=3.0" datasets accelerate pyvi
import torch, transformers, sentence_transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| sentence-transformers", sentence_transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (turn on the GPU accelerator)")

In [ ]:
# 3. Unit tests (about 20 seconds): checks that the environment runs the code correctly
!python -m pytest -q tests 2>&1 | tail -3

In [ ]:
# 4. Settings. WORK = work_full: the final collection (6,337 songs) shipped in the bundle.
WORK = "work_full"
QUICK = False          # True = 2 small models, 1 seed, 300 training pairs (about 10 minutes), to test the setup
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"

In [ ]:
# 5. All experiments: lexical runs (CPU), 5 zero-shot bi-encoders, NA-FT (2 bases x 3 seeds + 2 ablations),
#    each bi-encoder also on diacritic-restored queries, hybrid fusion, cross-encoder reranking, report. Each finished step is skipped if you run the cell again.
flag = "--quick" if QUICK else ""
!python scripts/run_gpu_experiments.py --work {WORK} {flag} 2>&1 | grep -v -E "it/s\]|s/it\]|Batches"

In [ ]:
# 6. Results table (test split; models were chosen on dev)
print(open(f"{WORK}/results/report_test.md").read())

In [ ]:
# 7. Package the small result files (no model weights, no vector files)
import json
out = "/kaggle/working/visongsearch_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob(f"{WORK}/results/*"):
        z.write(path, os.path.relpath(path, WORK))
    for path in glob.glob(f"{WORK}/runs/*.qids"):
        z.write(path, os.path.relpath(path, WORK))
    for path in glob.glob(f"{WORK}/dense/*/search_stats*.json") + glob.glob(f"{WORK}/models/*/naft_info.json"):
        z.write(path, os.path.relpath(path, WORK))
    for path in glob.glob(f"{WORK}/dense/*/meta.json"):
        meta = json.load(open(path))
        small = {k: v for k, v in meta.items() if k not in ("pids", "docs")}
        z.writestr(os.path.relpath(path, WORK), json.dumps(small))
print(out, round(os.path.getsize(out) / 1e6, 1), "MB")

Xong. Tải `visongsearch_results.zip` ở tab *Output* và gửi vào thread của project. File chứa bảng kết quả, thứ hạng từng truy vấn của mọi hệ thống (để kiểm định thống kê), trọng số trộn đã chọn trên dev, thời gian chạy và phiên bản thư viện.